First, I simply import the data and review the structure to identify the problem

In [140]:
import pandas as pd

df = pd.read_csv('broken_hero.csv')
df.head()

,ID,Full_Name,Age,Gender,Height_cm,Weight_kg,Occupation,City,Join_Date,Salary,Email
0,1,Johnathan Doe,25,Male,175.0,72.5,Engineer,New York,2020-01-15,75000,john.doe@example.com
1,2,Jane Smith,30,Female,165.0,63.5,Doctor,Los Angeles,2020-02-20,120000,jane_smith@domain.com
2,3,Emily Johnson,-5,Female,168.0,54.4,Artist,New York,2020-03-10,NaN,emily.johnson[at]example.com
3,4,Michael Brown,185,Male,185.0,81.6,Chef,Chicago,2020-01-05,55000,michael.brown@example.com
4,5,John Doe,25,Male,175.0,72.5,Engineer,New York,2020-01-15,75000,john.doe@example.com


In [141]:
df.info()
df.describe(include='all')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          300 non-null    int64  
 1   Full_Name   300 non-null    object 
 2   Age         300 non-null    int64  
 3   Gender      300 non-null    object 
 4   Height_cm   299 non-null    float64
 5   Weight_kg   300 non-null    float64
 6   Occupation  300 non-null    object 
 7   City        299 non-null    object 
 8   Join_Date   299 non-null    object 
 9   Salary      298 non-null    object 
 10  Email       299 non-null    object 
dtypes: float64(2), int64(2), object(7)
memory usage: 25.9+ KB


,ID,Full_Name,Age,Gender,Height_cm,Weight_kg,Occupation,City,Join_Date,Salary,Email
count,300.000000,300,300.000000,300,299.000000,300.000000,300,299,299,298,299
unique,NaN,159,NaN,4,NaN,NaN,50,45,52,29,157
top,NaN,Green Lantern,NaN,Male,NaN,NaN,Hero,New York,2020-01-01,70000,catwoman@gotham.com
freq,NaN,6,NaN,227,NaN,NaN,117,112,27,41,6
mean,150.500000,NaN,141.643333,NaN,179.474916,79.044667,NaN,NaN,NaN,NaN,NaN
std,86.746758,NaN,368.875616,NaN,12.645736,22.303940,NaN,NaN,NaN,NaN,NaN
min,1.000000,NaN,-5.000000,NaN,120.000000,45.000000,NaN,NaN,NaN,NaN,NaN
25%,75.750000,NaN,30.000000,NaN,171.000000,70.000000,NaN,NaN,NaN,NaN,NaN
50%,150.500000,NaN,35.000000,NaN,180.000000,75.000000,NaN,NaN,NaN,NaN,NaN
75%,225.250000,NaN,50.000000,NaN,185.000000,85.000000,NaN,NaN,NaN,NaN,NaN


In [142]:
df.isna().sum()

ID            0
Full_Name     0
Age           0
Gender        0
Height_cm     1
Weight_kg     0
Occupation    0
City          1
Join_Date     1
Salary        2
Email         1
dtype: int64

In [143]:
df.duplicated().sum()

np.int64(0)

Now, after analysing the data in this way, I can see anomalies, incorrect values and missing values in the cells.
Next, I will work on the columns to bring the dataset into proper version

In [144]:
df.dtypes

ID              int64
Full_Name      object
Age             int64
Gender         object
Height_cm     float64
Weight_kg     float64
Occupation     object
City           object
Join_Date      object
Salary         object
Email          object
dtype: object

I am listing the data types in the numeric columns, problematic records are converted to NaN.

In [145]:
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['Height_cm'] = pd.to_numeric(df['Height_cm'], errors='coerce')
df['Weight_kg'] = pd.to_numeric(df['Weight_kg'], errors='coerce')
df['Salary'] = pd.to_numeric(df['Salary'], errors='coerce')
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce')
df.dtypes

ID                     int64
Full_Name             object
Age                    int64
Gender                object
Height_cm            float64
Weight_kg            float64
Occupation            object
City                  object
Join_Date     datetime64[ns]
Salary               float64
Email                 object
dtype: object

Now replacing empty values. I will fill in the median

In [146]:
df.isna().sum()

ID            0
Full_Name     0
Age           0
Gender        0
Height_cm     1
Weight_kg     0
Occupation    0
City          1
Join_Date     2
Salary        3
Email         1
dtype: int64

In [147]:
df['Height_cm'] = df['Height_cm'].fillna(df['Height_cm'].median())
df['City'] = df['City'].fillna('Unknown')
df['Join_Date'] = df['Join_Date'].fillna('Unknown')
df['Salary'] = df['Salary'].fillna(df['Salary'].median())
df['Email'] = df['Email'].fillna('Unknown')
df.isna().sum()

ID            0
Full_Name     0
Age           0
Gender        0
Height_cm     0
Weight_kg     0
Occupation    0
City          0
Join_Date     0
Salary        0
Email         0
dtype: int64

During the initial inspection, I noticed that the age data contained negative values, so now the right-hand side of the columns has correct values.

In [148]:
df[df['Age'] < 0]

,ID,Full_Name,Age,Gender,Height_cm,Weight_kg,Occupation,City,Join_Date,Salary,Email
2,3,Emily Johnson,-5,Female,168.0,54.4,Artist,New York,2020-03-10 00:00:00,80000.0,emily.johnson[at]example.com


In [149]:
df.loc[df['Age'] < 0, 'Age'] = df['Age'].median()
df[df['Age'] < 0]

,ID,Full_Name,Age,Gender,Height_cm,Weight_kg,Occupation,City,Join_Date,Salary,Email


In [150]:
for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique()[:10])

Full_Name ['Johnathan Doe' 'Jane Smith' 'Emily Johnson' 'Michael Brown' 'John Doe'
 'Alice Williams' 'Chris Evans' 'Mary Jane' 'Robert Johnson' 'Julia White']
Gender ['Male' 'Female' 'male' 'MALE']
Occupation ['Engineer' 'Doctor' 'Artist' 'Chef' 'Houston' 'Nurse' 'Teacher'
 'Designer' 'Scientist' 'Manager']
City ['New York' 'Los Angeles' 'Chicago' '2020-04-01' 'Unknown' 'San Francisco'
 'Houston' 'Tatooine' 'Pallet Town' 'Azalea Town']
Join_Date [Timestamp('2020-01-15 00:00:00') Timestamp('2020-02-20 00:00:00')
 Timestamp('2020-03-10 00:00:00') Timestamp('2020-01-05 00:00:00')
 'Unknown' Timestamp('2020-05-15 00:00:00')
 Timestamp('2020-03-25 00:00:00') Timestamp('2020-06-05 00:00:00')
 Timestamp('2020-01-30 00:00:00') Timestamp('2020-02-15 00:00:00')]
Email ['john.doe@example.com' 'jane_smith@domain.com'
 'emily.johnson[at]example.com' 'michael.brown@example.com' 'Unknown'
 'chris.evans@domain.com' 'mary.jane@school.edu' 'robert.j@domain.com'
 'julia.white@domain.com' 'david.smith@@ex

Here I can see anomaly in object type records, I will remove extra spaces and put them all to the same format

In [151]:
df['City'] = df['City'].str.strip().str.title()
df['Full_Name'] = df['Full_Name'].str.strip().str.title()
df['Gender'] = df['Gender'].str.strip().str.title()
df['Occupation'] = df['Occupation'].str.strip().str.title()
df['Join_Date'] = pd.to_datetime(df['Join_Date'], errors='coerce').dt.strftime('%Y-%m-%d')
df['Email'] = df['Email'].str.strip().str.lower()

for col in df.select_dtypes(include='object').columns:
    print(col, df[col].unique()[:10])

Full_Name ['Johnathan Doe' 'Jane Smith' 'Emily Johnson' 'Michael Brown' 'John Doe'
 'Alice Williams' 'Chris Evans' 'Mary Jane' 'Robert Johnson' 'Julia White']
Gender ['Male' 'Female']
Occupation ['Engineer' 'Doctor' 'Artist' 'Chef' 'Houston' 'Nurse' 'Teacher'
 'Designer' 'Scientist' 'Manager']
City ['New York' 'Los Angeles' 'Chicago' '2020-04-01' 'Unknown' 'San Francisco'
 'Houston' 'Tatooine' 'Pallet Town' 'Azalea Town']
Join_Date ['2020-01-15' '2020-02-20' '2020-03-10' '2020-01-05' nan '2020-05-15'
 '2020-03-25' '2020-06-05' '2020-01-30' '2020-02-15']
Email ['john.doe@example.com' 'jane_smith@domain.com'
 'emily.johnson[at]example.com' 'michael.brown@example.com' 'unknown'
 'chris.evans@domain.com' 'mary.jane@school.edu' 'robert.j@domain.com'
 'julia.white@domain.com' 'david.smith@@example.com']


Now I have notice that Join_Date have some problem with values so I want to check them 

In [152]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 11 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   ID          300 non-null    int64  
 1   Full_Name   300 non-null    object 
 2   Age         300 non-null    int64  
 3   Gender      300 non-null    object 
 4   Height_cm   300 non-null    float64
 5   Weight_kg   300 non-null    float64
 6   Occupation  300 non-null    object 
 7   City        300 non-null    object 
 8   Join_Date   298 non-null    object 
 9   Salary      300 non-null    float64
 10  Email       300 non-null    object 
dtypes: float64(3), int64(2), object(6)
memory usage: 25.9+ KB


,ID,Full_Name,Age,Gender,Height_cm,Weight_kg,Occupation,City,Join_Date,Salary,Email
0,1,Johnathan Doe,25,Male,175.0,72.5,Engineer,New York,2020-01-15,75000.0,john.doe@example.com
1,2,Jane Smith,30,Female,165.0,63.5,Doctor,Los Angeles,2020-02-20,120000.0,jane_smith@domain.com
2,3,Emily Johnson,35,Female,168.0,54.4,Artist,New York,2020-03-10,80000.0,emily.johnson[at]example.com
3,4,Michael Brown,185,Male,185.0,81.6,Chef,Chicago,2020-01-05,55000.0,michael.brown@example.com
4,5,John Doe,25,Male,175.0,72.5,Engineer,New York,2020-01-15,75000.0,john.doe@example.com


In [153]:
df.describe(include='all')

,ID,Full_Name,Age,Gender,Height_cm,Weight_kg,Occupation,City,Join_Date,Salary,Email
count,300.000000,300,300.000000,300,300.000000,300.000000,300,300,298,300.000000,300
unique,NaN,159,NaN,2,NaN,NaN,50,46,51,NaN,157
top,NaN,Green Lantern,NaN,Male,NaN,NaN,Hero,New York,2020-01-01,NaN,catwoman@gotham.com
freq,NaN,6,NaN,229,NaN,NaN,117,112,27,NaN,6
mean,150.500000,NaN,141.776667,NaN,179.476667,79.044667,NaN,NaN,NaN,90723.333333,NaN
std,86.746758,NaN,368.829660,NaN,12.624608,22.303940,NaN,NaN,NaN,39434.889255,NaN
min,1.000000,NaN,5.000000,NaN,120.000000,45.000000,NaN,NaN,NaN,0.000000,NaN
25%,75.750000,NaN,30.000000,NaN,171.500000,70.000000,NaN,NaN,NaN,70000.000000,NaN
50%,150.500000,NaN,35.000000,NaN,180.000000,75.000000,NaN,NaN,NaN,80000.000000,NaN
75%,225.250000,NaN,50.000000,NaN,185.000000,85.000000,NaN,NaN,NaN,100000.000000,NaN


Something wrong with Join_Date, still have missing values, replace them to 'unknown'

In [154]:
df['Join_Date'] = df['Join_Date'].fillna('Unknown')

Work with Emails
I am creating a new column where there will be a mark indicating whether the address corresponds to the email format.
Then output all rows that have suspicious values and change value manualy where it's just typo and wrong format

In [155]:
pattern = r"^[A-Za-z0-9\._%+\-]+@[A-Za-z0-9\.\-]+\.[A-Za-z]{2,}$"
df['Email_valid'] = df['Email'].astype(str).str.match(pattern)
df[['Email', 'Email_valid']]

,Email,Email_valid
0,john.doe@example.com,True
1,jane_smith@domain.com,True
2,emily.johnson[at]example.com,False
3,michael.brown@example.com,True
4,john.doe@example.com,True
...,...,...
295,doctor.fate@nabu.com,True
296,ghost.rider@example.com,True
297,blade@example.com,True
298,morpheus@thematrix.com,True


In [156]:
invalid_emails = df[~df['Email_valid']]
invalid_emails[['Email']]

,Email
2,emily.johnson[at]example.com
5,unknown
10,david.smith@@example.com


In [157]:
df.loc[2, 'Email'] = 'emily.johnson@example.com'
df.loc[10, 'Email'] = 'david.smith@example.com'

Save dataset

In [158]:
df.to_csv("clean_hero.csv", index=False)